<a href="https://colab.research.google.com/github/gitaukennedy/-Fine-Tune-BERT-for-Sentiment-Analysis/blob/main/Fine_tunedBERT_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch scikit-learn


In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized = dataset.map(tokenize, batched=True)

In [ ]:
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # 👈 reduce for speed
    weight_decay=0.01,
    logging_dir="./logs"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"].shuffle(seed=42).select(range(2000)),
    eval_dataset=tokenized["test"].select(range(500))
)

In [ ]:
train_small = tokenized["train"].select(range(1000))
test_small = tokenized["test"].select(range(200))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=test_small
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
import torch

text = "This product is absolutely fantastic!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits)

print("Sentiment:", "Positive" if prediction == 1 else "Negative")

FULL EVALUATION


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1
    }

NIMMEKA A TRAINER WITH *METRICS*

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=test_small,
    compute_metrics=compute_metrics
)

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

predictions = trainer.predict(test_small)
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

This project involves fine-tuning a BERT (Bidirectional Encoder Representations from Transformers) model to perform sentiment analysis on text data, enabling it to classify inputs as positive or negative based on contextual understanding.

 After training on a labeled dataset, the model is evaluated using key performance metrics such as accuracy, F1-score, and a confusion matrix, which together provide insight into its overall performance, balance between precision and recall, and types of classification errors.

  The model can also be tested on custom user inputs to demonstrate real-world usability.
  
   Such systems are widely applied across industries including customer experience analytics, social media monitoring, financial sentiment tracking, and automated support systems, where extracting meaning and opinion from large volumes of text is essential for informed decision-making.